# 05 — Layer 3b: Temporal Alignment + Patient Identity

🔧 Script

Two more Layer 3 sub-tasks: (1) temporal alignment, especially the Mandel rule for DocumentReferences; (2) patient identity resolution across sources via Fellegi-Sunter probabilistic linkage.


## Section 1 — Temporal alignment

### 1a. The Mandel rule on DocumentReference

In [ ]:
from pathlib import Path
from ehi_atlas.harmonize.temporal import clinical_time

# Build a DocumentReference with only the metadata date set (NOT context.period.start)
docref_metadata_only = {
    "resourceType": "DocumentReference",
    "id": "docref-metadata-test",
    "date": "2025-11-01T09:00:00Z",   # ← this is INDEX time, not clinical time
    "status": "current",
    "type": {"coding": [{"system": "http://loinc.org", "code": "11506-3"}]},
}

ct = clinical_time(docref_metadata_only)
print("DocumentReference with ONLY date set:")
print("  timestamp:", ct.timestamp)
print("  confidence:", ct.confidence)
print("  source_field:", ct.source_field)
print()
print("→ The Mandel rule: docRef.date is NOT used as clinical time (confidence='uncertain')")


In [ ]:
# Now add context.period.start — this IS clinical time
docref_with_context = {
    "resourceType": "DocumentReference",
    "id": "docref-context-test",
    "date": "2025-11-01T09:00:00Z",
    "status": "current",
    "context": {
        "period": {
            "start": "2026-01-15",   # ← this is the clinical encounter date
        }
    },
    "type": {"coding": [{"system": "http://loinc.org", "code": "11506-3"}]},
}

ct = clinical_time(docref_with_context)
print("DocumentReference with context.period.start set:")
print("  timestamp:", ct.timestamp)
print("  confidence:", ct.confidence)
print("  source_field:", ct.source_field)


### 1b. normalize_bundle_temporal on a small bundle

In [ ]:
from ehi_atlas.harmonize.temporal import normalize_bundle_temporal, EXT_CLINICAL_TIME

# Build a mini bundle with two resources
mini_bundle = {
    "resourceType": "Bundle",
    "type": "collection",
    "entry": [
        {"resource": {
            "resourceType": "Observation",
            "id": "obs-creatinine",
            "status": "final",
            "effectiveDateTime": "2025-09-12",
            "code": {"coding": [{"system": "http://loinc.org", "code": "2160-0"}]},
            "valueQuantity": {"value": 1.4, "unit": "mg/dL"},
        }},
        {"resource": {
            "resourceType": "Condition",
            "id": "cond-hyperlipidemia",
            "onsetDateTime": "2020-03-01",
            "code": {"coding": [{"system": "http://snomed.info/sct", "code": "55822004"}]},
        }},
    ],
}

normalize_bundle_temporal(mini_bundle)

for entry in mini_bundle["entry"]:
    res = entry["resource"]
    exts = res.get("meta", {}).get("extension", [])
    ct_ext = next((e for e in exts if e.get("url") == EXT_CLINICAL_TIME), None)
    if ct_ext:
        print(f"{res['resourceType']}/{res['id']} → clinical_time = {ct_ext.get('valueDateTime') or ct_ext.get('valueString')}")


## Section 2 — Patient identity resolution

### 2a. Build two PatientFingerprints for Rhett759 from different sources

In [ ]:
from ehi_atlas.harmonize.identity import PatientFingerprint, score

fp_synthea = PatientFingerprint(
    source="synthea",
    local_patient_id="rhett759-synthea",
    family_name="Rohan584",
    given_names=("Rhett759",),
    birth_date="1966-09-14",
    gender="male",
    address_zip="01001",
    mrn_value="rhett759-mrn-synthea",
    mrn_system="synthea://mrn",
)

fp_epic = PatientFingerprint(
    source="epic-ehi",
    local_patient_id="RHETT759",
    family_name="Rohan584",
    given_names=("Rhett759",),
    birth_date="1966-09-14",
    gender="male",
    address_zip=None,
    mrn_value="MRN-EPIC-RHETT",
    mrn_system="urn:epic:mrn",
)

match_score = score(fp_synthea, fp_epic)
print("Fellegi-Sunter aggregate score:", round(match_score.aggregate, 4))
print("Decision:", match_score.decision)
print()
print("Component scores:")
print("  name:   ", round(match_score.name, 4))
print("  dob:    ", round(match_score.dob, 4))
print("  address:", round(match_score.address, 4))
print("  gender: ", round(match_score.gender, 4))


### 2b. build_identity_index → one canonical record

In [ ]:
from ehi_atlas.harmonize.identity import build_identity_index, merged_patient_resource

index = build_identity_index(
    [fp_synthea, fp_epic],
    canonical_id_for={
        fp_synthea.local_patient_id: "rhett759",
        fp_epic.local_patient_id:    "rhett759",
    },
)

canon = index.canonical_patients["rhett759"]
print("Canonical ID:", canon.canonical_id)
print("Contributing sources:", [fp.source for fp in canon.fingerprints])
merged_pat = merged_patient_resource(canon)
print("Merged Patient.id:", merged_pat["id"])
print("Merged identifiers:")
for ident in merged_pat.get("identifier", [])[:4]:
    print(f"  {ident.get('system')}: {ident.get('value')}")


**Next:** [06_layer3_condition_merge_artifact_1.ipynb](./06_layer3_condition_merge_artifact_1.ipynb)